## Week 6 — "The Price Is Right" Capstone (mmaitsimwale)

### What Week 6 covers
- **Day 1:** Data curation — Amazon Reviews 2023 dataset scraped into `Item` objects (title, category, price, weight, summary).
- **Day 2:** Data pre-processing — LLMs rewrite messy product descriptions into clean `item.summary` text.
- **Day 3:** Evaluation harness (`pricer.evaluator.Tester`) + baselines (random, average, linear regression, Random Forest).
- **Day 4:** Frontier LLMs as zero-shot pricers — GPT-4.1-nano, Claude Opus, Gemini 2.5 Flash, Grok, GPT-5.
- **Day 5:** Fine-tuning — upload JSONL to OpenAI, train a private GPT-4.1-nano variant, evaluate.

### Exercise task: Multi-Provider LLM Price Estimation Benchmark

The capstone asks: *how accurately can a language model estimate a product's price from its description?*

This notebook benchmarks **two provider categories**:
- **OpenRouter** (paid models) — routes to OpenAI GPT variants
- **Groq** (free OSS models) — runs Llama 3.3, Llama 3.1, DeepSeek R1 Distill

Every model receives the same zero-shot prompt. Results are measured via the standard `Tester` harness
(average absolute error in dollars + R²) and ranked in a final leaderboard table.


## What we are measuring

**Task:** given `item.summary` (a text description of a product), predict its price in USD.

**Metric:** average absolute error (`$`) over `BENCHMARK_SIZE` test items. Lower is better.
A secondary metric is R² — closer to 100 % means predictions correlate with true prices.

**Providers / models under test:**

| Provider | Model | Type |
|----------|-------|------|
| OpenRouter | `openai/gpt-4.1-nano` | Paid (small, fast) |
| OpenRouter | `openai/gpt-4o-mini` | Paid (balanced) |
| Groq | `llama-3.3-70b-versatile` | Free OSS (large) |
| Groq | `llama-3.1-8b-instant` | Free OSS (small, fast) |
| Groq | `deepseek-r1-distill-llama-70b` | Free OSS (reasoning) |

**Credentials used:** `OR_API_KEY` + `OR_CLIENT_URL` for OpenRouter; `GROQ_API_KEY` + `GROQ_CLIENT_URL` for Groq.
Both APIs are OpenAI-compatible, so we use `openai.OpenAI(api_key=..., base_url=...)`.


In [ ]:
# Setup: imports, repo-root resolution, pricer module path
import os
import re
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import r2_score
import pandas as pd


def _repo_root() -> Path:
    """Walk up from cwd until week6/pricer is found."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "week6" / "pricer").is_dir():
            return cand
    raise RuntimeError(
        "Cannot find week6/pricer — start Jupyter from the llm_engineering repo root."
    )


REPO_ROOT = _repo_root()

# Add week6/ to sys.path so `from pricer.items import Item` works from any CWD
WEEK6_PATH = str(REPO_ROOT / "week6")
if WEEK6_PATH not in sys.path:
    sys.path.insert(0, WEEK6_PATH)

from pricer.items import Item
from pricer.evaluator import Tester

BENCHMARK_SIZE = 50   # items per model — raise to 200 for a more reliable run
WORKERS = 3

print(f"Repo root : {REPO_ROOT}")
print(f"Week6 path: {WEEK6_PATH}")
print(f"Benchmark : {BENCHMARK_SIZE} items per model, {WORKERS} workers")


In [ ]:
# .env / key check — same style as week4/day5.ipynb and week5/day5.ipynb
load_dotenv(REPO_ROOT / ".env", override=True)

hf_token         = os.getenv("HF_TOKEN")
openai_api_key   = os.getenv("OPENAI_API_KEY")
or_api_key       = os.getenv("OR_API_KEY")
or_client_url    = os.getenv("OR_CLIENT_URL", "https://openrouter.ai/api/v1")
groq_api_key     = os.getenv("GROQ_API_KEY")
groq_client_url  = os.getenv("GROQ_CLIENT_URL", "https://api.groq.com/openai/v1")

for name, val in [
    ("HF_TOKEN",        hf_token),
    ("OPENAI_API_KEY",  openai_api_key),
    ("OR_API_KEY",      or_api_key),
    ("GROQ_API_KEY",    groq_api_key),
]:
    if val:
        print(f"{name} exists and begins {val[:8]}")
    else:
        print(f"{name} NOT SET — some models will be unavailable")


In [ ]:
# Two OpenAI-compatible clients — one per provider (same pattern as week4/day5.ipynb)

# OpenRouter: routes to paid frontier models (GPT-4.1-nano, GPT-4o-mini, ...)
or_client = OpenAI(
    api_key=or_api_key,
    base_url=or_client_url,
)

# Groq: fast inference for free OSS models (Llama, DeepSeek, Mixtral, ...)
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url=groq_client_url,
)

print("OpenRouter client  :", or_client.base_url)
print("Groq client        :", groq_client.base_url)


In [ ]:
# Load the curated Amazon product dataset from HuggingFace (Day 1 data)
# Using the "lite" variant (~22k train / 1k val / 1k test) to keep cost low.
# Switch to "ed-donner/items_full" for a more representative benchmark.

from huggingface_hub import login

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN — cannot load dataset")

DATASET = "ed-donner/items_lite"

train, val, test = Item.from_hub(DATASET)
print(f"Loaded  {len(train):,} train | {len(val):,} val | {len(test):,} test items")
print(f"Sample  : {test[0]}")
print(f"Summary : {test[0].summary[:120]}...")


In [ ]:
# Prompt construction — identical to day4.ipynb cell 20
# A single zero-shot user message: describe the product, ask for a price.

def messages_for(item: Item) -> list[dict]:
    """Build the messages list for zero-shot price estimation."""
    return [
        {
            "role": "user",
            "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            ),
        }
    ]


# Sanity-check on the first test item
print(messages_for(test[0]))
print(f"\nActual price: ${test[0].price:.2f}")


In [ ]:
# Pricer factory — builds a named pricing function for any OpenAI-compatible client.
# The Tester class derives the display title from the function name, so we use
# a dynamic closure so each pricer has a unique __name__.

def make_pricer(client: OpenAI, model: str, max_tokens: int = 10):
    """
    Return a callable `fn(item) -> str` that calls `client` with `model`.

    The function name is set to the model slug so Tester.make_title() renders it nicely.
    """
    def pricer(item: Item) -> str:
        response = client.chat.completions.create(
            model=model,
            messages=messages_for(item),
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    # Give the function a descriptive name for the Tester title
    pricer.__name__ = model.replace("/", "_").replace("-", "_").replace(".", "_")
    pricer.__qualname__ = pricer.__name__
    return pricer


# ── OpenRouter (paid) ────────────────────────────────────────────────────────
gpt_4_1_nano_or = make_pricer(or_client,   "openai/gpt-4.1-nano")
gpt_4o_mini_or  = make_pricer(or_client,   "openai/gpt-4o-mini")

# ── Groq (free OSS) ──────────────────────────────────────────────────────────
llama_33_70b        = make_pricer(groq_client, "llama-3.3-70b-versatile")
llama_31_8b         = make_pricer(groq_client, "llama-3.1-8b-instant")
deepseek_r1_distill = make_pricer(groq_client, "deepseek-r1-distill-llama-70b")

# Registry: display name → (function, provider)
PRICERS: dict[str, tuple] = {
    "GPT-4.1-nano (OpenRouter)":     (gpt_4_1_nano_or,    "OpenRouter"),
    "GPT-4o-mini (OpenRouter)":      (gpt_4o_mini_or,     "OpenRouter"),
    "Llama-3.3-70b (Groq)":          (llama_33_70b,       "Groq"),
    "Llama-3.1-8b (Groq)":           (llama_31_8b,        "Groq"),
    "DeepSeek-R1-Distill (Groq)":    (deepseek_r1_distill,"Groq"),
}

print(f"Registered {len(PRICERS)} pricers:")
for display_name, (fn, provider) in PRICERS.items():
    print(f"  [{provider}] {display_name}")


In [ ]:
# Run the benchmark — one Tester pass per model.
# Tester runs BENCHMARK_SIZE items concurrently (WORKERS threads), prints
# colour-coded per-item errors, then shows error-trend + scatter charts.

benchmark_results: dict[str, dict] = {}

for display_name, (fn, provider) in PRICERS.items():
    print(f"\n{'='*60}")
    print(f"  {display_name}")
    print(f"{'='*60}")
    try:
        t = Tester(fn, test, title=display_name, size=BENCHMARK_SIZE, workers=WORKERS)
        t.run()
        avg_err = sum(t.errors) / len(t.errors)
        r2      = r2_score(t.truths, t.guesses) * 100
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": avg_err,
            "r2": r2,
            "errors": t.errors,
            "guesses": t.guesses,
            "truths": t.truths,
        }
    except Exception as exc:
        print(f"  ERROR: {exc}")
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": float("inf"),
            "r2": float("-inf"),
            "errors": [],
            "guesses": [],
            "truths": [],
        }

print(f"\nBenchmark complete. {len(benchmark_results)} models evaluated.")


In [ ]:
# Results leaderboard — ranked by average absolute error (lower = better)

rows = []
for display_name, r in benchmark_results.items():
    if r["avg_error"] == float("inf"):
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": "ERROR",
            "R² (%)": "—",
        }
    else:
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": f"{r['avg_error']:.2f}",
            "R² (%)": f"{r['r2']:.1f}",
        }
    rows.append(row)

# Sort: models with errors go last, others sorted by avg_error ascending
def sort_key(row):
    try:
        return float(row["Avg Error ($)"])
    except (ValueError, TypeError):
        return float("inf")

rows.sort(key=sort_key)

print(f"\n{'='*70}")
print(f"{'LEADERBOARD':^70}")
print(f"{'='*70}")
print(f"{'Rank':<5} {'Model':<35} {'Provider':<12} {'Avg Err ($)':>12} {'R²':>8}")
print("-" * 70)

for rank, row in enumerate(rows, 1):
    err = row["Avg Error ($)"]
    r2  = row["R² (%)"]
    print(f"{rank:<5} {row['Model']:<35} {row['Provider']:<12} {err:>12} {r2:>8}")

print("-" * 70)

# Best model summary
if rows and rows[0]["Avg Error ($)"] != "ERROR":
    winner = rows[0]
    print(f"\nBest model : {winner['Model']}")
    print(f"Provider   : {winner['Provider']}")
    print(f"Avg Error  : ${winner['Avg Error ($)']}")
    print(f"R²         : {winner['R² (%)']}")


## Key Findings

- **OpenRouter** routes calls to official OpenAI endpoints; GPT-4o-mini generally balances cost and accuracy well for structured numerical tasks.
- **Groq** provides blazing-fast inference for open-weight models; Llama-3.3-70b is typically the strongest OSS option for price estimation.
- **DeepSeek-R1-Distill** (reasoning model) may overthink a simple regression task and sometimes produces verbose output — the Tester strips the first number it finds, but watch for parsing failures.
- The Day 5 fine-tuned `gpt-4.1-nano` variant (trained on 100 examples from this same dataset) consistently outperforms zero-shot models, illustrating the value of task-specific fine-tuning even with minimal data.

### Next steps (Day 5 direction)
To fine-tune on this data, convert training items to JSONL and upload to OpenAI (or your provider of choice):


In [ ]:
# Day 5 preview: JSONL format for fine-tuning (no upload — just show the format)

def make_jsonl(items: list, n: int = 5) -> str:
    """Produce JSONL fine-tuning data in OpenAI chat format."""
    lines = []
    for item in items[:n]:
        messages = [
            {"role": "user", "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            )},
            {"role": "assistant", "content": f"${item.price:.2f}"},
        ]
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)


sample_jsonl = make_jsonl(train, n=5)
print("=== Sample fine-tuning JSONL (first 5 items) ===\n")
print(sample_jsonl)
print(f"\n... and {len(train):,} more training examples available.")
print("\nTo fine-tune, upload via:")
print('  openai.files.create(open("fine_tune_train.jsonl","rb"), purpose="fine-tune")')
print('  openai.fine_tuning.jobs.create(training_file=..., model="gpt-4.1-nano-2025-04-14")')
